# Lab: Support Vector Machines

Based on the Chapter 9 lab (sections 9.6.1-9.6.5). We fit support vector classifiers and SVMs to simulated data,
recover their decision function by hand from the fitted support vectors, and finally apply them to the `OJ`
dataset (the reference lab uses the `Khan` gene expression data; `OJ` is the dataset of exercise 9.8 of the book).

Cells marked `# TODO` must be completed.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

The function below draws the predicted class, the boundary $f(x) = 0$ (solid), the margins $f(x) = \pm 1$ (dashed)
and circles the support vectors.

In [ ]:
def plot_svm(svc, X, y, ax=None, h=0.02, pad=0.5):
    ax = ax or plt.gca()
    x1 = np.arange(X[:, 0].min() - pad, X[:, 0].max() + pad, h)
    x2 = np.arange(X[:, 1].min() - pad, X[:, 1].max() + pad, h)
    g1, g2 = np.meshgrid(x1, x2)
    grid = np.c_[g1.ravel(), g2.ravel()]
    ax.contourf(g1, g2, svc.predict(grid).reshape(g1.shape), alpha=0.2, cmap="brg" if len(svc.classes_) > 2 else "bwr")
    if len(svc.classes_) == 2:
        f = svc.decision_function(grid).reshape(g1.shape)
        ax.contour(g1, g2, f, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=1)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="brg" if len(svc.classes_) > 2 else "bwr", edgecolor="k", s=30)
    sv = svc.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors="none", edgecolors="k")
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title=f"{len(sv)} support vectors")

## 1. Support vector classifier

### 1.1 Simulated data

20 observations with two features, labels $-1$ and $1$; class 1 is shifted by 1 in both coordinates. The classes
overlap.

In [ ]:
rng = np.random.default_rng(21)
X = rng.standard_normal((20, 2))
y = np.repeat([-1, 1], 10)
X[y == 1] += 1
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k", s=40)
plt.show()

Fit `SVC(kernel="linear", C=10)` and plot it. Then refit with `C=0.1`.

Note: in `scikit-learn`, `C` is the **cost** of a margin violation, the opposite of the budget $C$ of the book.
Smaller `C` means a wider margin and more support vectors.

In [ ]:
# TODO
svc1 = ...
svc2 = ...

### 1.2 The decision function by hand

For a linear kernel the fitted function is $f(x) = \beta_0 + \beta^\top x$, with `coef_` $= \beta$ and
`intercept_` $= \beta_0$. The dual solution gives the same $\beta$ as a combination of the support vectors only:

$$\beta = \sum_{i \in S} \alpha_i y_i x_i,$$

where `dual_coef_` stores $\alpha_i y_i$ for the support vectors $i \in S$.

For `svc1`:
1. recover $\beta$ from `dual_coef_` and `support_vectors_`, and compare with `coef_`;
2. check that `decision_function(X)` equals $X\beta + \beta_0$;
3. compute $y_i f(x_i)$ for every point. Which points have $y_i f(x_i) \le 1$? Compare with `support_`;
4. compute the margin width $2 / \|\beta\|$.

In [ ]:
# TODO

- Check that $0 < \alpha_i \le C$ for all support vectors (`np.abs(dual_coef_)`), and that $\sum_i \alpha_i y_i = 0$.
  Which support vectors have $\alpha_i = C$? Where are they with respect to the margin?

In [ ]:
# TODO

### 1.3 Choosing C by cross-validation

Use `GridSearchCV` with 10-fold CV over `C` in `[0.001, 0.01, 0.1, 1, 5, 10, 100]`. Then generate a test set from
the same distribution (200 points) and report the confusion matrix of the best model.

In [ ]:
# TODO

In [ ]:
X_test = rng.standard_normal((200, 2))
y_test = np.repeat([-1, 1], 100)
X_test[y_test == 1] += 1
# TODO

- With only 20 training points, how much do you trust the value of `C` chosen by CV? (Look at how flat the CV
  scores are.)

In [ ]:
X_sep = rng.standard_normal((20, 2))
y_sep = np.repeat([-1, 1], 10)
X_sep[y_sep == 1] += 3
# TODO

- How many support vectors are there? What is $y_i f(x_i)$ for them?
- Refit with `C=0.1`. The training data are still separated, but is the margin the same? Which classifier would you
  expect to generalize better?

In [ ]:
X = rng.standard_normal((200, 2))
X[:100] += 2
X[100:150] -= 2
y = np.r_[np.full(150, -1), np.ones(50, int)]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=2)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k", s=30)
plt.show()

Fit an SVM with radial kernel, `gamma=1`, `C=1`, plot it on the training data and compute the test confusion matrix.
Then repeat with `C=100`.

In [ ]:
# TODO
svm1 = ...
svm2 = ...

- Which model is more flexible? Which one overfits?

In [ ]:
def rbf_kernel(A, B, gamma):
    # TODO
    ...


# TODO

### 2.3 Choosing C and gamma

10-fold CV over `C` in `[0.01, 0.1, 1, 10, 100]` and `gamma` in `[0.5, 1, 2, 3, 4]`. Report the best pair and its
test accuracy.

In [ ]:
# TODO

## 3. ROC curves

An SVM has no probabilities, but `decision_function` gives a score: classify as 1 when $f(x) > t$ and move $t$.
Compare two radial SVMs, `gamma=2` and `gamma=50` (`C=1`), with ROC curves on the training and on the test set.

In [ ]:
svm3 = SVC(kernel="rbf", gamma=2, C=1).fit(X_train, y_train)
svm4 = SVC(kernel="rbf", gamma=50, C=1).fit(X_train, y_train)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (name, Xs, ys) in zip(ax, [("training data", X_train, y_train), ("test data", X_test, y_test)]):
    # TODO: ROC curve and AUC of svm3 and svm4 on (Xs, ys)
    a.plot([0, 1], [0, 1], "k:")
    a.set(xlabel="false positive rate", ylabel="true positive rate", title=name)
    a.legend(loc="lower right")
plt.show()

- Which model looks better on the training data? And on the test data? Explain with the bias-variance trade-off.

## 4. More than two classes

Add a third class (label 0) around $(0, 6)$. `SVC` fits one SVM per pair of classes (one-versus-one) and votes.

In [ ]:
X3 = np.vstack([X, rng.standard_normal((50, 2)) + [0, 6]])
y3 = np.r_[y, np.zeros(50, int)]
svm5 = SVC(kernel="rbf", C=10, gamma=1).fit(X3, y3)
plot_svm(svm5, X3, y3)
plt.show()

`decision_function_shape="ovo"` returns the score of each of the $\binom{3}{2}$ pairwise SVMs; `"ovr"` (default)
aggregates them into one score per class. Print the shape of `decision_function` in both cases.

In [ ]:
# TODO

## 5. Application: the OJ dataset

1070 purchases of orange juice: `Purchase` is `CH` (Citrus Hill) or `MM` (Minute Maid), and 17 features describe the
prices, discounts and customer brand loyalty (`LoyalCH`). Encode `Store7` as 0/1 and use 800 purchases for training.

In [ ]:
oj = pd.read_csv("datasets/OJ.csv")
oj["Store7"] = (oj["Store7"] == "Yes").astype(int)
X_oj = oj.drop(columns="Purchase")
y_oj = oj["Purchase"]
Xo_train, Xo_test, yo_train, yo_test = train_test_split(X_oj, y_oj, train_size=800, random_state=0)
print(y_oj.value_counts())
oj.head()

### 5.1 Scaling

The radial kernel depends on distances, so a feature measured in large units dominates. Compare the test accuracy
of `SVC(kernel="rbf")` with and without standardization (`make_pipeline(StandardScaler(), SVC(...))`).

In [ ]:
# TODO

### 5.2 Linear, radial and polynomial kernels

For each kernel (linear; radial; polynomial of degree 2), use a standardized pipeline and:
1. fit with `C=0.01` and report the number of support vectors, train and test error;
2. choose `C` in `np.logspace(-2, 1, 8)` by 10-fold CV and report the train and test error of the best model.

Pipeline parameters are addressed as `"svc__C"`.

In [ ]:
kernels = {"linear": {"kernel": "linear"}, "radial": {"kernel": "rbf"}, "poly (d = 2)": {"kernel": "poly", "degree": 2}}
rows = []
# TODO
pd.DataFrame(rows).set_index("kernel").round(3)

- With `C=0.01`, why are so many points support vectors?
- Which kernel would you choose? Is the non-linear flexibility worth it on this dataset?

## 6. To go further

- `src/models.py` implements an SVM from scratch (the SMO algorithm on the dual problem). Run `pytest tests` and
  `python main.py`, and compare the support vectors with those of `scikit-learn`.
- Compare the best SVM on `OJ` with logistic regression and a random forest.